## 1. Load features.csv

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()

# Nếu notebook đang nằm trong thư mục notebooks/, quay về project root
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FEATURE_PATH = PROJECT_ROOT / "data" / "processed" / "features.csv"
REPORT_DIR = PROJECT_ROOT / "outputs" / "feature_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

ID_COL = "CONS_NO"
TARGET_COL = "FLAG"

features = pd.read_csv(FEATURE_PATH)

feature_cols = [
    col for col in features.columns
    if col not in [ID_COL, TARGET_COL]
]

print("Feature path:", FEATURE_PATH)
print("Shape:", features.shape)
print("Số feature:", len(feature_cols))
print("Target distribution:")
print(features[TARGET_COL].value_counts(normalize=True))

display(features.head())

Feature path: d:\CODE\CS114----ML-Project---Energy-Theft-Detection\data\processed\features.csv
Shape: (42372, 163)
Số feature: 161
Target distribution:
FLAG
0    0.914684
1    0.085316
Name: proportion, dtype: float64


,CONS_NO,FLAG,mean_consumption,median_consumption,std_consumption,min_consumption,max_consumption,sum_consumption,q25_consumption,q75_consumption,...,recent30_to_total_mean_ratio,zero_to_low_ratio,missing_x_zero_ratio,missing_x_low_ratio,missing_x_recent_drop,outlier_x_volatility,outlier_x_max_change,drop_x_recent_drop,drop_x_segment_drop,missing_x_outlier
0,0387DD8A07E07FDA6271170F86AD9151,1,0.951103,0.000,3.100520,0.0,18.29,983.44037,0.000,0.0000,...,0.379559,0.936147,0.003236,0.003457,-0.001192,0.0,0.0,-0.000895,0.010831,0.0
1,01D6177B5D4FFE0CABA9EF17DAFC2B84,1,4.100145,1.060,6.608953,0.0,44.57,4239.55030,0.430,4.8075,...,4.096276,0.130719,0.010550,0.080710,-0.504287,0.0,0.0,-0.000000,0.000000,0.0
2,4B75AC4F2D8434CFF62DB64D0BB43103,1,10.036695,7.275,11.302323,0.0,79.71,10377.94200,2.585,12.5075,...,0.547126,0.288235,0.014666,0.050881,0.074958,0.0,0.0,0.010317,0.227716,0.0
3,B32AC8CC6D5D805AC053557AB05F5343,1,3.786828,0.000,8.056552,0.0,40.98,3915.58080,0.000,0.0000,...,4.414423,1.000000,0.012720,0.012720,-0.047260,0.0,0.0,-0.002783,-0.000000,0.0
4,EDFC78B07BA2908B3395C4EB2304665E,1,61.553967,54.630,40.909782,0.0,399.40,63646.80000,54.630,54.6300,...,0.023708,0.690909,0.024133,0.034929,2.288539,0.0,0.0,0.020242,0.049658,0.0


## 2. Check missing, infinity và duplicate columns
- NaN: giá trị bị thiếu.
- inf hoặc -inf: thường xuất hiện khi chia cho 0.
- duplicate columns: trùng tên cột.

In [2]:
X = features[feature_cols]

n_missing = X.isna().sum().sum()
n_inf = np.isinf(X.to_numpy(dtype=np.float64)).sum()
n_duplicate_cols = features.columns.duplicated().sum()

print("Missing values:", n_missing)
print("Infinity values:", n_inf)
print("Duplicate column names:", n_duplicate_cols)

if n_missing > 0:
    missing_report = X.isna().sum()
    missing_report = missing_report[missing_report > 0].sort_values(ascending=False)
    display(missing_report.head(30))

if n_inf > 0:
    inf_mask = np.isinf(X.to_numpy(dtype=np.float64))
    inf_cols = X.columns[inf_mask.any(axis=0)].tolist()
    print("Columns with inf:", inf_cols[:30])

Missing values: 0
Infinity values: 0
Duplicate column names: 0


## 3. Check constant features
Feature constant là feature có cùng một giá trị cho tất cả khách hàng.
Ví dụ một cột toàn 0 hoặc toàn 1.

In [3]:
constant_cols = []

for col in feature_cols:
    if features[col].nunique(dropna=False) <= 1:
        constant_cols.append(col)

print("Số constant features:", len(constant_cols))

constant_df = pd.DataFrame({
    "feature": constant_cols,
    "unique_values": [features[col].unique().tolist() for col in constant_cols]
})

display(constant_df.head(50))

constant_df.to_csv(REPORT_DIR / "constant_features.csv", index=False)
print("Saved:", REPORT_DIR / "constant_features.csv")

Số constant features: 2


,feature,unique_values
0,negative_count_raw,[0]
1,negative_ratio_raw,[0.0]


Saved: d:\CODE\CS114----ML-Project---Energy-Theft-Detection\outputs\feature_reports\constant_features.csv


## 4. Check near-constant features

Near-constant feature là feature gần như chỉ có một giá trị chiếm áp đảo.
Ví dụ:
- 99.9% giá trị là 0.
- Chỉ một vài dòng khác biệt.

In [4]:
near_constant_rows = []

for col in feature_cols:
    value_ratio = features[col].value_counts(normalize=True, dropna=False).iloc[0]
    
    if value_ratio >= 0.995:
        near_constant_rows.append({
            "feature": col,
            "top_value_ratio": value_ratio,
            "n_unique": features[col].nunique(dropna=False),
            "top_value": features[col].value_counts(dropna=False).index[0],
        })

near_constant_df = pd.DataFrame(near_constant_rows)
near_constant_df = near_constant_df.sort_values("top_value_ratio", ascending=False)

print("Số near-constant features:", len(near_constant_df))
display(near_constant_df.head(50))

near_constant_df.to_csv(REPORT_DIR / "near_constant_features.csv", index=False)
print("Saved:", REPORT_DIR / "near_constant_features.csv")

Số near-constant features: 5


,feature,top_value_ratio,n_unique,top_value
0,negative_count_raw,1.000000,1,0.0
1,negative_ratio_raw,1.000000,1,0.0
2,is_all_missing_raw,0.999882,2,0.0
4,max_consumption_raw_missing,0.999882,2,0.0
3,is_very_high_missing_raw,0.995398,2,0.0


Saved: d:\CODE\CS114----ML-Project---Energy-Theft-Detection\outputs\feature_reports\near_constant_features.csv


## 5. Compare features by target FLAG

So sánh trung bình feature giữa hai nhóm:
- FLAG = 0: normal
- FLAG = 1: theft

In [5]:
compare_rows = []

for col in feature_cols:
    normal_values = features.loc[features[TARGET_COL] == 0, col]
    theft_values = features.loc[features[TARGET_COL] == 1, col]

    normal_mean = normal_values.mean()
    theft_mean = theft_values.mean()

    normal_median = normal_values.median()
    theft_median = theft_values.median()

    diff_mean = theft_mean - normal_mean
    abs_diff_mean = abs(diff_mean)

    ratio_mean = theft_mean / (normal_mean + 1e-6)

    compare_rows.append({
        "feature": col,
        "normal_mean": normal_mean,
        "theft_mean": theft_mean,
        "diff_mean": diff_mean,
        "abs_diff_mean": abs_diff_mean,
        "ratio_mean": ratio_mean,
        "normal_median": normal_median,
        "theft_median": theft_median,
    })

feature_compare_df = pd.DataFrame(compare_rows)

feature_compare_df = feature_compare_df.sort_values(
    "abs_diff_mean",
    ascending=False
)

display(feature_compare_df.head(50))

feature_compare_df.to_csv(REPORT_DIR / "feature_target_compare.csv", index=False)
print("Saved:", REPORT_DIR / "feature_target_compare.csv")

,feature,normal_mean,theft_mean,diff_mean,abs_diff_mean,ratio_mean,normal_median,theft_median
5,sum_consumption,7755.404497,21178.168854,13422.764357,13422.764357,2.730763,5424.747600,8677.547000
133,max_consumption_raw,137.402956,427.572383,290.169427,290.169427,3.111814,22.020000,38.020000
18,low_consumption_count,287.824728,184.069433,-103.755296,103.755296,0.639519,43.000000,32.000000
21,max_low_streak_clean,196.481023,102.646196,-93.834826,93.834826,0.522423,14.000000,10.000000
16,zero_count_clean,190.055345,118.356017,-71.699328,71.699328,0.622745,7.000000,4.000000
124,missing_count_raw,259.495188,325.386722,65.891534,65.891534,1.253922,96.000000,221.000000
134,max_missing_streak_raw,238.977991,304.675242,65.697251,65.697251,1.274909,82.000000,192.000000
20,max_zero_streak_clean,127.582991,66.856155,-60.726837,60.726837,0.524021,2.000000,2.000000
126,zero_count_raw,141.340738,85.936376,-55.404362,55.404362,0.608009,6.000000,3.000000
4,max_consumption,33.547820,70.859894,37.312074,37.312074,2.112206,22.020000,38.020000


Saved: d:\CODE\CS114----ML-Project---Energy-Theft-Detection\outputs\feature_reports\feature_target_compare.csv


## 6. Correlation with target
Vì `FLAG` là 0/1, correlation ở đây giúp xem feature nào có xu hướng tăng khi FLAG = 1.

Lưu ý:
- Correlation cao không đảm bảo model tốt.
- Correlation thấp không có nghĩa feature vô dụng, vì quan hệ có thể phi tuyến.
- Nhưng đây là một cách kiểm tra nhanh feature nào có tín hiệu tuyến tính với target.

In [6]:
target_corr = features[feature_cols + [TARGET_COL]].corr(numeric_only=True)[TARGET_COL]
target_corr = target_corr.drop(TARGET_COL)

target_corr_df = target_corr.reset_index()
target_corr_df.columns = ["feature", "corr_with_flag"]
target_corr_df["abs_corr_with_flag"] = target_corr_df["corr_with_flag"].abs()

target_corr_df = target_corr_df.sort_values(
    "abs_corr_with_flag",
    ascending=False
)

display(target_corr_df.head(50))

target_corr_df.to_csv(REPORT_DIR / "target_correlation.csv", index=False)
print("Saved:", REPORT_DIR / "target_correlation.csv")

,feature,corr_with_flag,abs_corr_with_flag
85,roll30_mean_range,0.239019,0.239019
93,roll90_mean_range,0.234072,0.234072
77,roll7_mean_range,0.233391,0.233391
74,roll7_mean_std,0.232916,0.232916
119,weekend_std,0.232909,0.232909
84,roll30_mean_max,0.229925,0.229925
82,roll30_mean_std,0.229654,0.229654
76,roll7_mean_max,0.229516,0.229516
2,std_consumption,0.227931,0.227931
105,segment_mean_range,0.227629,0.227629


Saved: d:\CODE\CS114----ML-Project---Energy-Theft-Detection\outputs\feature_reports\target_correlation.csv


## 7. High correlation between features
Nếu hai feature tương quan quá cao, có thể chúng đang mang thông tin gần trùng nhau.


In [7]:
CORR_THRESHOLD = 0.98

corr_matrix = features[feature_cols].corr(numeric_only=True).abs()

upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_pairs = []

for col in upper_triangle.columns:
    correlated_features = upper_triangle.index[upper_triangle[col] >= CORR_THRESHOLD].tolist()
    
    for row in correlated_features:
        high_corr_pairs.append({
            "feature_1": row,
            "feature_2": col,
            "abs_corr": upper_triangle.loc[row, col],
        })

high_corr_df = pd.DataFrame(high_corr_pairs)

if len(high_corr_df) > 0:
    high_corr_df = high_corr_df.sort_values("abs_corr", ascending=False)

print(f"Số cặp feature có correlation >= {CORR_THRESHOLD}:", len(high_corr_df))
display(high_corr_df.head(100))

high_corr_df.to_csv(REPORT_DIR / "high_correlation_pairs.csv", index=False)
print("Saved:", REPORT_DIR / "high_correlation_pairs.csv")

Số cặp feature có correlation >= 0.98: 138


,feature_1,feature_2,abs_corr
1,mean_consumption,sum_consumption,1.000000
7,zero_count_clean,zero_ratio_clean,1.000000
130,zero_count_raw,zero_ratio_total_raw,1.000000
133,outlier_count_raw,outlier_ratio_raw,1.000000
136,is_all_missing_raw,max_consumption_raw_missing,1.000000
...,...,...,...
11,mean_consumption,first_half_mean,0.984648
39,first_half_mean,roll7_mean_avg,0.984641
52,first_half_mean,roll30_mean_avg,0.984611
127,std_consumption,weekend_std,0.984609


Saved: d:\CODE\CS114----ML-Project---Energy-Theft-Detection\outputs\feature_reports\high_correlation_pairs.csv


## 8. Suggested drop columns

In [8]:
suggested_drop_cols = []

# Chỉ tự động đề xuất drop constant features
suggested_drop_cols.extend(constant_cols)

suggested_drop_cols = sorted(set(suggested_drop_cols))

suggested_drop_df = pd.DataFrame({
    "feature": suggested_drop_cols,
    "reason": "constant_feature",
})

print("Số feature đề xuất drop:", len(suggested_drop_df))
display(suggested_drop_df)

suggested_drop_df.to_csv(REPORT_DIR / "suggested_drop_features.csv", index=False)
print("Saved:", REPORT_DIR / "suggested_drop_features.csv")

Số feature đề xuất drop: 2


,feature,reason
0,negative_count_raw,constant_feature
1,negative_ratio_raw,constant_feature


Saved: d:\CODE\CS114----ML-Project---Energy-Theft-Detection\outputs\feature_reports\suggested_drop_features.csv
